<a href="https://colab.research.google.com/github/gonzalesmclarry/FinalnaGuroni/blob/main/VILLARTE_TEXT_FEATURE_EXTRACTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Load the Dataset (assuming the file was already uploaded in a previous session)
# If the file is not found, you may need to re-upload it using files.upload()
# The filename from previous upload was 'Fin_lab-PRProject_dataset (2).csv'
# Ensure this filename matches your uploaded file.
filename = 'Fin_lab-PRProject_dataset (2).csv'
print(f'Using previously uploaded file: "{filename}"')


Using previously uploaded file: "Fin_lab-PRProject_dataset (2).csv"


In [ ]:
# Dataset first 5 rows
import pandas as pd

df = pd.read_csv(filename)
df.head()

,Unnamed: 0,recommendationid,language,review,Reaction
0,0,77057085,english,Is good. Do play.,0
1,1,77052689,english,AAAAAAAA,0
2,2,77049252,english,Fun game,1
3,3,77049089,english,"Great game, worth every penny!",0
4,4,35101272,english,Like,0


In [ ]:
import nltk
from nltk.corpus import stopwords
import re

# Download the stopwords corpus if not already downloaded
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    if not isinstance(text, str):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and digits
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    words = text.split()
    # Remove stopwords
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

# Apply the function to the 'review' column and create a new 'review_cleaned' column
df['review_cleaned'] = df['review'].apply(remove_stopwords)

print("Stop words removed. A new column 'review_cleaned' has been added to the DataFrame.")
print(df[['review', 'review_cleaned']].head())

Stop words removed. A new column 'review_cleaned' has been added to the DataFrame.
                           review                review_cleaned
0               Is good. Do play.                     good play
1                        AAAAAAAA                      aaaaaaaa
2                        Fun game                      fun game
3  Great game, worth every penny!  great game worth every penny
4                            Like                          like


In [ ]:
import pandas as pd

# Ensure all words in 'review_cleaned' are lowercase
# Note: The previous step (remove_stopwords function) already converts text to lowercase.
# This step explicitly applies lowercase conversion again to the 'review_cleaned' column.
df['review_cleaned'] = df['review_cleaned'].str.lower()

print("All words in 'review_cleaned' column are now explicitly ensured to be in lowercase.")
print(df[['review', 'review_cleaned']].head())

All words in 'review_cleaned' column are now explicitly ensured to be in lowercase.
                           review                review_cleaned
0               Is good. Do play.                     good play
1                        AAAAAAAA                      aaaaaaaa
2                        Fun game                      fun game
3  Great game, worth every penny!  great game worth every penny
4                            Like                          like


In [ ]:
import nltk
from nltk.stem import PorterStemmer

# Initialize Porter Stemmer
stemmer = PorterStemmer()

def apply_stemming(text):
    if not isinstance(text, str):
        return ""
    # Tokenize the text into words
    words = text.split()
    # Apply stemming to each word
    stemmed_words = [stemmer.stem(word) for word in words]
    # Join the stemmed words back into a sentence
    return ' '.join(stemmed_words)

# Apply the stemming function to the 'review_cleaned' column
df['review_stemmed'] = df['review_cleaned'].apply(apply_stemming)

print("Basic stemming applied. A new column 'review_stemmed' has been added to the DataFrame.")
print(df[['review_cleaned', 'review_stemmed']].head())

Basic stemming applied. A new column 'review_stemmed' has been added to the DataFrame.
                 review_cleaned                review_stemmed
0                     good play                     good play
1                      aaaaaaaa                      aaaaaaaa
2                      fun game                      fun game
3  great game worth every penny  great game worth everi penni
4                          like                          like


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Initialize TfidfVectorizer
# Using 'review_stemmed' as it contains the preprocessed text
# max_features can be used to limit vocabulary size if still experiencing memory issues
vectorizer = TfidfVectorizer()

# Fit and transform the stemmed reviews to create the TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(df['review_stemmed'])

# Get the vocabulary (unique terms)
vocabulary = vectorizer.get_feature_names_out()
print("\n--- Vocabulary (first 20 terms): ---")
print(vocabulary[:20])
print(f"Total unique terms in vocabulary: {len(vocabulary)}")

# Note: For large datasets, it's generally NOT recommended to convert
# the entire sparse matrix to a dense DataFrame for memory efficiency.
# We will keep it as a sparse matrix (tfidf_matrix).
# If you need to inspect a small portion, you can do so like this:
# tfidf_df = pd.DataFrame(tfidf_matrix[:5, :10].toarray(), columns=vocabulary[:10])

print(f"\nTF-IDF matrix created with shape: {tfidf_matrix.shape}")
print("TF-IDF matrix is stored as a sparse matrix to save memory.")
print("You can inspect small portions or use it directly for modeling.")


--- Vocabulary (first 20 terms): ---
['aa' 'aaa' 'aaaaa' 'aaaaaaa' 'aaaaaaaa' 'aaaaaaaaaaaaaaa'
 'aaaaaaaaaaaaaaaa' 'aaaaaaaaaaaaaaaaaaa' 'aaaaaaaaaaaaaaaaaaaaaaaaaaa'
 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'
 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'
 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'
 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'
 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa

In [ ]:
import numpy as np
from nltk.stem import PorterStemmer
import re

stemmer = PorterStemmer()

words_to_analyze_original = ['business', 'making', 'support', 'data', 'system']

def stem_word(word):
    return stemmer.stem(word)

# Stem the words to match the vocabulary used in TfidfVectorizer
words_to_analyze_stemmed = [stem_word(word) for word in words_to_analyze_original]

# Create a mapping from stemmed word to its index in the vocabulary
word_to_idx = {word: idx for idx, word in enumerate(vocabulary)}

output_markdown = """### TF, IDF, and TF-IDF Calculations for Specific Words\n\nLet's break down the calculations for the words: **{}**.\n\n""".format(', '.join(words_to_analyze_original))

# Get the total number of documents from the original dataframe
N_docs = df.shape[0]

for original_word, stemmed_word in zip(words_to_analyze_original, words_to_analyze_stemmed):
    output_markdown += f"#### Word: '{original_word}' (Stemmed: '{stemmed_word}')\n\n"

    if stemmed_word not in word_to_idx:
        output_markdown += f"*   '{original_word}' (stemmed as '{stemmed_word}') is not in the vocabulary generated by TF-IDF. Therefore, TF, IDF, and TF-IDF values will be 0.\n\n"
        output_markdown += f"**1. Term Frequency (TF)**: 0.0\n"
        output_markdown += f"**2. Inverse Document Frequency (IDF)**: 0.0\n"
        output_markdown += f"**3. TF-IDF**: 0.0\n\n"
        continue

    term_idx = word_to_idx[stemmed_word]
    idf_value = vectorizer.idf_[term_idx]

    # Find the first document where the stemmed word appears
    doc_indices = df[df['review_stemmed'].str.contains(r'\b' + re.escape(stemmed_word) + r'\b', na=False, regex=True)].index.tolist()

    if not doc_indices:
        output_markdown += f"*   The word '{original_word}' (stemmed as '{stemmed_word}') does not appear in any document.\n\n"
        output_markdown += f"**1. Term Frequency (TF)**: 0.0 (as it doesn't appear in any document)\n"
        output_markdown += f"**2. Inverse Document Frequency (IDF)**: {idf_value:.4f} (calculated from global document frequencies)\n"
        output_markdown += f"**3. TF-IDF**: 0.0 (TF is 0)\n\n"
        continue

    # Pick the first document where the word appears
    sample_doc_idx = doc_indices[0]
    sample_doc_text = df['review_stemmed'].iloc[sample_doc_idx]

    # Calculate raw term frequency (count of word in sample document)
    words_in_sample_doc = sample_doc_text.split()
    raw_count = words_in_sample_doc.count(stemmed_word)
    total_words_in_sample_doc = len(words_in_sample_doc)

    # Calculate Term Frequency (TF) for the sample document
    tf_value = raw_count / total_words_in_sample_doc if total_words_in_sample_doc > 0 else 0.0

    # Get TF-IDF value from the sparse matrix by direct indexing
    # Sparse matrices return 0 for non-stored (zero) elements by default when indexed.
    tfidf_val_from_matrix = tfidf_matrix[sample_doc_idx, term_idx]

    output_markdown += f"*   For illustration, let's consider Document {sample_doc_idx} where '{original_word}' appears.\n"
    output_markdown += f"    (Sample document text: '{sample_doc_text[:100]}...')\n\n"

    output_markdown += f"**1. Term Frequency (TF)**\n"
    output_markdown += f"    *   Count of '{stemmed_word}' in Document {sample_doc_idx}: `{raw_count}`\n"
    output_markdown += f"    *   Total words in Document {sample_doc_idx}: `{total_words_in_sample_doc}`\n"
    output_markdown += f"    *   TF for '{stemmed_word}' in Document {sample_doc_idx}: `{raw_count} / {total_words_in_sample_doc} = {tf_value:.4f}`\n\n"

    output_markdown += f"**2. Inverse Document Frequency (IDF)**\n"
    output_markdown += f"    *   The `TfidfVectorizer` IDF formula (with `smooth_idf=True` by default) is: `log((1 + N_docs) / (1 + df(t))) + 1`\n"
    output_markdown += f"    *   Total Documents (N_docs): `{N_docs}`\n"
    # df_t (document frequency of the term) is not directly exposed by vectorizer once fitted,
    # but vectorizer.idf_ attribute provides the final IDF values.
    output_markdown += f"    *   IDF for '{stemmed_word}': `{idf_value:.4f}`\n\n"

    output_markdown += f"**3. TF-IDF**\n"
    output_markdown += f"    *   TF-IDF is TF * IDF.\n"
    output_markdown += f"    *   Calculated TF-IDF for '{stemmed_word}' in Document {sample_doc_idx}: `{tf_value:.4f} * {idf_value:.4f} = {(tf_value * idf_value):.4f}`\n"
    output_markdown += f"    *   TF-IDF value from `tfidf_matrix` for '{stemmed_word}' in Document {sample_doc_idx}: `{tfidf_val_from_matrix:.4f}`\n\n"


### TF, IDF, and TF-IDF Calculations for Specific Words

Break down of the calculations for the words: **business, making, support, data, system**.

#### Word: 'business' (Stemmed: 'busi')

*   For illustration, let's consider Document 11 where 'business' appears.
    (Sample document text: 'play hour final reach point play game someth els best game make hand busi listen lectur audiobook po...')

**1. Term Frequency (TF)**
    *   Count of 'busi' in Document 11: `1`
    *   Total words in Document 11: `20`
    *   TF for 'busi' in Document 11: `1 / 20 = 0.0500`

**2. Inverse Document Frequency (IDF)**
    *   The `TfidfVectorizer` IDF formula (with `smooth_idf=True` by default) is: `log((1 + N_docs) / (1 + df(t))) + 1`
    *   Total Documents (N_docs): `46742`
    *   IDF for 'busi': `8.6547`

**3. TF-IDF**
    *   TF-IDF is TF * IDF.
    *   Calculated TF-IDF for 'busi' in Document 11: `0.0500 * 8.6547 = 0.4327`
    *   TF-IDF value from `tfidf_matrix` for 'busi' in Document 11: `0.4327`

#### Word: 'making' (Stemmed: 'make')

*   For illustration, let's consider Document 26 where 'making' appears.
    (Sample document text: 'love thi game alway make come back play')

**1. Term Frequency (TF)**
    *   Count of 'make' in Document 26: `1`
    *   Total words in Document 26: `8`
    *   TF for 'make' in Document 26: `1 / 8 = 0.1250`

**2. Inverse Document Frequency (IDF)**
    *   The `TfidfVectorizer` IDF formula (with `smooth_idf=True` by default) is: `log((1 + N_docs) / (1 + df(t))) + 1`
    *   Total Documents (N_docs): `46742`
    *   IDF for 'make': `6.0097`

**3. TF-IDF**
    *   TF-IDF is TF * IDF.
    *   Calculated TF-IDF for 'make' in Document 26: `0.1250 * 6.0097 = 0.7512`
    *   TF-IDF value from `tfidf_matrix` for 'make' in Document 26: `0.7512`

#### Word: 'support' (Stemmed: 'support')

*   For illustration, let's consider Document 3154 where 'support' appears.
    (Sample document text: 'game good need more languag support')

**1. Term Frequency (TF)**
    *   Count of 'support' in Document 3154: `1`
    *   Total words in Document 3154: `5`
    *   TF for 'support' in Document 3154: `1 / 5 = 0.2000`

**2. Inverse Document Frequency (IDF)**
    *   The `TfidfVectorizer` IDF formula (with `smooth_idf=True` by default) is: `log((1 + N_docs) / (1 + df(t))) + 1`
    *   Total Documents (N_docs): `46742`
    *   IDF for 'support': `7.3392`

**3. TF-IDF**
    *   TF-IDF is TF * IDF.
    *   Calculated TF-IDF for 'support' in Document 3154: `0.2000 * 7.3392 = 1.4678`
    *   TF-IDF value from `tfidf_matrix` for 'support' in Document 3154: `1.4678`

#### Word: 'data' (Stemmed: 'data')

*   The word 'data' (stemmed as 'data') does not appear in any document.

**1. Term Frequency (TF)**: 0.0 (as it doesn't appear in any document)
**2. Inverse Document Frequency (IDF)**: 10.1258 (calculated from global document frequencies)
**3. TF-IDF**: 0.0 (TF is 0)

#### Word: 'system' (Stemmed: 'system')

*   For illustration, let's consider Document 113 where 'system' appears.
    (Sample document text: 'dont count keeper realli fun game spend hundr hour get bore combin skill dodg aim luck get cool item synerg keeper charact piec utter dogcrap unlock garbag garbag way win break game abus system suck')

**1. Term Frequency (TF)**
    *   Count of 'system' in Document 113: `1`
    *   Total words in Document 113: `35`
    *   TF for 'system' in Document 113: `1 / 35 = 0.0286`

**2. Inverse Document Frequency (IDF)**
    *   The `TfidfVectorizer` IDF formula (with `smooth_idf=True` by default) is: `log((1 + N_docs) / (1 + df(t))) + 1`
    *   Total Documents (N_docs): `46742`
    *   IDF for 'system': `7.1120`

**3. TF-IDF**
    *   TF-IDF is TF * IDF.
    *   Calculated TF-IDF for 'system' in Document 113: `0.0286 * 7.1120 = 0.2034`
    *   TF-IDF value from `tfidf_matrix` for 'system' in Document 113: `0.2034`


### Interpretation of TF, IDF, and TF-IDF Results

TF-IDF (Term Frequency-Inverse Document Frequency) is a numerical statistic that reflects how important a word is to a document in a collection or corpus. It is often used as a weighting factor in information retrieval and text mining. A high TF-IDF value means that the word is frequent in that specific document (high TF) but also rare across the entire collection of documents (high IDF), suggesting it is highly relevant to that document's topic.

Analyzing the words:

*   **'support' (Stemmed: 'support')**:
    *   **TF-IDF Value**: `1.4678` (in Document 3154)
    *   **Interpretation**: 'Support' has the highest TF-IDF among the analyzed words. This indicates that 'support' is a very important and distinguishing term in Document 3154. The sample text "game good need more languag support" strongly suggests that this document's main focus is about the need for or presence of 'support', likely related to language or technical aspects within the game.

*   **'making' (Stemmed: 'make')**:
    *   **TF-IDF Value**: `0.7512` (in Document 26)
    *   **Interpretation**: 'Making' has a significant TF-IDF value, indicating its relevance to Document 26. The sample text "love thi game alway make come back play" highlights that the document's core message revolves around the *action of making* (or causing) something, possibly related to engaging gameplay that makes players return.

*   **'business' (Stemmed: 'busi')**:
    *   **TF-IDF Value**: `0.4327` (in Document 11)
    *   **Interpretation**: The TF-IDF for 'business' is lower than 'support' and 'making'. While 'business' is present and contributes to the document's meaning, it might be part of a broader context rather than the sole primary focus. The sample document "play hour final reach point play game someth els best game make hand busi listen lectur audiobook po..." suggests 'business' is one element among several activities or considerations mentioned in the review.

*   **'system' (Stemmed: 'system')**:
    *   **TF-IDF Value**: `0.2034` (in Document 113)
    *   **Interpretation**: 'System' has a relatively low TF-IDF value. Although it appears in Document 113, its low value suggests that while it's mentioned, it's not a primary defining term for this specific document compared to other words. The document is quite long and covers various aspects of gameplay, and 'system' is likely a supporting detail rather than the main topic.

*   **'data' (Stemmed: 'data')**:
    *   **TF-IDF Value**: `0.0`
    *   **Interpretation**: The TF-IDF value of 0 for 'data' indicates that this word (in its stemmed form) did not appear in any of the documents in our preprocessed dataset. Therefore, it holds no importance or relevance to any specific document in this corpus based on this analysis.